# Titanic Survival Prediction

Complete end-to-end binary classification project: data cleaning, EDA, feature engineering, model training, and evaluation, using the uploaded `Titanic-Dataset.csv`.

In [ ]:
"""
TITANIC SURVIVAL PREDICTION - LEVEL 1(a)
Complete end-to-end ML classification project.
Runs from raw CSV to trained model, evaluation, and cleaned outputs.
"""



In [ ]:
# ======================================================================
# 1. IMPORT LIBRARIES
# ======================================================================
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

CHART_DIR = "charts"
os.makedirs(CHART_DIR, exist_ok=True)

RANDOM_STATE = 42



In [ ]:
# ======================================================================
# 2 & 3. LOCATE + LOAD DATASET
# ======================================================================
DATA_PATH = "Titanic-Dataset.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH)

print("=" * 70)
print("SECTION 6: DATASET LOADED")
print("=" * 70)
print(f"File: {DATA_PATH}")
print(f"Rows: {df_raw.shape[0]}")
print(f"Columns: {df_raw.shape[1]}")
print("\nFirst 5 rows:\n", df_raw.head())
print("\nLast 5 rows:\n", df_raw.tail())
print("\nColumn names:", df_raw.columns.tolist())
print("\nData types:\n", df_raw.dtypes)

df = df_raw.copy()



In [ ]:
# ======================================================================
# 4. INITIAL DATASET ANALYSIS
# ======================================================================
print("\n" + "=" * 70)
print("SECTION 7: INITIAL ANALYSIS")
print("=" * 70)
print("Shape:", df.shape)
print("\nDescriptive statistics:\n", df.describe(include="all").T)

target_counts = df["Survived"].value_counts().sort_index()
n_not_survived = int(target_counts.get(0, 0))
n_survived = int(target_counts.get(1, 0))
total = len(df)
pct_survived = n_survived / total * 100
pct_not_survived = n_not_survived / total * 100

print(f"\nSurvived (1): {n_survived}  ({pct_survived:.2f}%)")
print(f"Did not survive (0): {n_not_survived}  ({pct_not_survived:.2f}%)")

plt.figure(figsize=(6, 5))
ax = sns.countplot(x="Survived", data=df, palette=["#c0392b", "#27ae60"])
ax.set_xticklabels(["Did Not Survive (0)", "Survived (1)"])
ax.set_title("Chart 1: Survival Distribution", fontsize=13, fontweight="bold")
ax.set_xlabel("Outcome")
ax.set_ylabel("Number of Passengers")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart1_survival_distribution.png")
plt.close()



In [ ]:
# ======================================================================
# 5. DATA QUALITY CHECK
# ======================================================================
print("\n" + "=" * 70)
print("SECTION 8-9: DATA QUALITY & MISSING VALUE ANALYSIS")
print("=" * 70)

missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": missing_count,
    "missing_pct": missing_pct,
    "n_unique": df.nunique(),
})
print(quality_summary)

n_duplicates = df.duplicated().sum()
print(f"\nExact duplicate rows: {n_duplicates}")

constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print(f"Constant columns: {constant_cols if constant_cols else 'None'}")

# ======================================================================
# INVALID VALUE CHECK (Section 13)
# ======================================================================
invalid_age = (df["Age"] < 0).sum()
invalid_fare = (df["Fare"] < 0).sum()
invalid_pclass = (~df["Pclass"].isin([1, 2, 3])).sum()
invalid_survived = (~df["Survived"].isin([0, 1])).sum()
invalid_sex = (~df["Sex"].isin(["male", "female"])).sum()
invalid_embarked = df["Embarked"].dropna().apply(lambda x: x not in ["S", "C", "Q"]).sum()

print("\nInvalid value checks:")
print(f"  Age < 0: {invalid_age}")
print(f"  Fare < 0: {invalid_fare}")
print(f"  Invalid Pclass: {invalid_pclass}")
print(f"  Invalid Survived: {invalid_survived}")
print(f"  Invalid Sex: {invalid_sex}")
print(f"  Invalid Embarked: {invalid_embarked}")



In [ ]:
# ======================================================================
# 6. MISSING VALUE TREATMENT (Section 10)
# ======================================================================
age_median = df["Age"].median()
df["Age"] = df["Age"].fillna(age_median)

embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)

# Cabin: extract Deck feature, mark missing as 'Unknown', then drop raw Cabin
df["Deck"] = df["Cabin"].apply(lambda x: x[0] if pd.notnull(x) else "Unknown")
df.drop(columns=["Cabin"], inplace=True)

print(f"\nAge missing values filled with median = {age_median}")
print(f"Embarked missing values filled with mode = '{embarked_mode}'")
print("Cabin -> engineered 'Deck' feature (first letter, 'Unknown' for missing); raw Cabin dropped.")
print("\nDeck value counts:\n", df["Deck"].value_counts())



In [ ]:
# ======================================================================
# 7. DUPLICATE HANDLING (Section 11)
# ======================================================================
rows_before = len(df)
df = df.drop_duplicates()
rows_after = len(df)
print(f"\nDuplicate removal: rows before = {rows_before}, rows after = {rows_after}")



In [ ]:
# ======================================================================
# 8. OUTLIER ANALYSIS (Section 14) - IQR method, informational only
# ======================================================================
print("\n" + "=" * 70)
print("SECTION 14: OUTLIER ANALYSIS (IQR METHOD)")
print("=" * 70)

outlier_report = {}
for col in ["Age", "Fare", "SibSp", "Parch"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_report[col] = {
        "Q1": q1, "Q3": q3, "IQR": iqr,
        "Lower Bound": lower, "Upper Bound": upper,
        "Num Outliers": n_outliers
    }
    print(f"\n{col}: Q1={q1:.2f} Q3={q3:.2f} IQR={iqr:.2f} "
          f"Lower={lower:.2f} Upper={upper:.2f} Outliers={n_outliers}")

outlier_df = pd.DataFrame(outlier_report).T
print("\nDecision: Outliers in Fare (high first-class fares) and SibSp/Parch (large")
print("families) are legitimate real-world observations, not data errors.")
print("No rows removed based on outlier detection.")

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, ["Age", "Fare", "SibSp", "Parch"]):
    sns.boxplot(y=df[col], ax=ax, color="#3498db")
    ax.set_title(col)
plt.suptitle("Outlier Check: Boxplots (Age, Fare, SibSp, Parch)", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/outlier_boxplots.png")
plt.close()



In [ ]:
# ======================================================================
# 9. FEATURE ENGINEERING (Sections 15-16)
# ======================================================================
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

# Title extraction
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
title_map_rare = df["Title"].value_counts()
rare_titles = title_map_rare[title_map_rare < 10].index.tolist()
df["Title"] = df["Title"].replace(rare_titles, "Rare")
df["Title"] = df["Title"].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})

print("\n" + "=" * 70)
print("SECTION 16: FEATURE ENGINEERING")
print("=" * 70)
print("Created: FamilySize = SibSp + Parch + 1")
print("Created: IsAlone = 1 if FamilySize == 1 else 0")
print("Created: Title extracted from Name, rare titles grouped as 'Rare'")
print("\nTitle distribution:\n", df["Title"].value_counts())



In [ ]:
# ======================================================================
# 10. EXPLORATORY DATA ANALYSIS (Sections 17-18)
# ======================================================================
print("\n" + "=" * 70)
print("SECTION 17-18: EDA")
print("=" * 70)

sex_survival = df.groupby("Sex")["Survived"].mean() * 100
print("\nSurvival rate by Sex (%):\n", sex_survival.round(2))

plt.figure(figsize=(6, 5))
ax = sns.barplot(x=sex_survival.index, y=sex_survival.values, palette="Set2")
ax.set_title("Chart 2: Survival Rate by Sex", fontsize=13, fontweight="bold")
ax.set_xlabel("Sex")
ax.set_ylabel("Survival Rate (%)")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart2_survival_by_sex.png")
plt.close()

pclass_survival = df.groupby("Pclass")["Survived"].mean() * 100
print("\nSurvival rate by Pclass (%):\n", pclass_survival.round(2))

plt.figure(figsize=(6, 5))
ax = sns.barplot(x=pclass_survival.index, y=pclass_survival.values, palette="Set3")
ax.set_title("Chart 3: Survival Rate by Passenger Class", fontsize=13, fontweight="bold")
ax.set_xlabel("Passenger Class")
ax.set_ylabel("Survival Rate (%)")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart3_survival_by_pclass.png")
plt.close()

plt.figure(figsize=(7, 5))
sns.histplot(data=df, x="Age", hue="Survived", multiple="stack", bins=30,
             palette=["#c0392b", "#27ae60"])
plt.title("Chart 4: Age Distribution by Survival", fontsize=13, fontweight="bold")
plt.xlabel("Age")
plt.ylabel("Count")
plt.legend(title="Survived", labels=["Survived", "Did Not Survive"])
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart4_age_by_survival.png")
plt.close()

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Survived", y="Fare", palette=["#c0392b", "#27ae60"])
plt.title("Chart 5: Fare Distribution by Survival", fontsize=13, fontweight="bold")
plt.xlabel("Survived (0 = No, 1 = Yes)")
plt.ylabel("Fare")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart5_fare_by_survival.png")
plt.close()

embarked_survival = df.groupby("Embarked")["Survived"].mean() * 100
print("\nSurvival rate by Embarked (%):\n", embarked_survival.round(2))

plt.figure(figsize=(6, 5))
ax = sns.barplot(x=embarked_survival.index, y=embarked_survival.values, palette="Set1")
ax.set_title("Chart 6: Survival Rate by Embarked Port", fontsize=13, fontweight="bold")
ax.set_xlabel("Embarked Port")
ax.set_ylabel("Survival Rate (%)")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart6_survival_by_embarked.png")
plt.close()

family_survival = df.groupby("FamilySize")["Survived"].mean() * 100
print("\nSurvival rate by FamilySize (%):\n", family_survival.round(2))

plt.figure(figsize=(7, 5))
ax = sns.barplot(x=family_survival.index, y=family_survival.values, palette="coolwarm")
ax.set_title("Chart 7: Survival Rate by Family Size", fontsize=13, fontweight="bold")
ax.set_xlabel("Family Size (incl. self)")
ax.set_ylabel("Survival Rate (%)")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart7_survival_by_familysize.png")
plt.close()



In [ ]:
# ======================================================================
# 11. FEATURE SELECTION
# ======================================================================
feature_cols = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked",
                 "FamilySize", "IsAlone", "Title"]
X = df[feature_cols].copy()
y = df["Survived"].copy()

numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize", "IsAlone"]
categorical_features = ["Pclass", "Sex", "Embarked", "Title"]



In [ ]:
# ======================================================================
# 12. TRAIN/TEST SPLIT (Section 19)
# ======================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("\n" + "=" * 70)
print("SECTION 19: TRAIN/TEST SPLIT")
print("=" * 70)
print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")



In [ ]:
# ======================================================================
# 13. PREPROCESSING PIPELINE (Section 20)
# ======================================================================
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])



In [ ]:
# ======================================================================
# 14. LOGISTIC REGRESSION MODEL (Section 21)
# ======================================================================
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
logreg_pipeline.fit(X_train, y_train)
y_pred_lr = logreg_pipeline.predict(X_test)
y_proba_lr = logreg_pipeline.predict_proba(X_test)[:, 1]



In [ ]:
# ======================================================================
# 15. OPTIONAL MODEL COMPARISON (Section 22)
# ======================================================================
dt_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=5))
])
dt_pipeline.fit(X_train, y_train)
y_pred_dt = dt_pipeline.predict(X_test)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE))
])
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

models = {
    "Logistic Regression": (logreg_pipeline, y_pred_lr),
    "Decision Tree": (dt_pipeline, y_pred_dt),
    "Random Forest": (rf_pipeline, y_pred_rf),
}

comparison_rows = []
for name, (pipe, preds) in models.items():
    comparison_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1-score": f1_score(y_test, preds),
    })
comparison_df = pd.DataFrame(comparison_rows).sort_values("Accuracy", ascending=False)

print("\n" + "=" * 70)
print("SECTION 22: MODEL COMPARISON")
print("=" * 70)
print(comparison_df.to_string(index=False))

# Final model = Logistic Regression (primary beginner-level model per assignment)
final_model_name = "Logistic Regression"
final_pipeline = logreg_pipeline
y_pred = y_pred_lr
y_proba = y_proba_lr



In [ ]:
# ======================================================================
# 16. MODEL EVALUATION (Sections 23-24)
# ======================================================================
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()
report = classification_report(y_test, y_pred, target_names=["Not Survived", "Survived"])

print("\n" + "=" * 70)
print(f"SECTION 23-24: FINAL MODEL EVALUATION ({final_model_name})")
print("=" * 70)
print(f"Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"\nConfusion Matrix:\nTN={tn}  FP={fp}\nFN={fn}  TP={tp}")
print("\nClassification Report:\n", report)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Survived", "Survived"],
            yticklabels=["Not Survived", "Survived"])
plt.title("Chart 8: Confusion Matrix - Logistic Regression", fontsize=13, fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart8_confusion_matrix.png")
plt.close()



In [ ]:
# ======================================================================
# 17. ROC-AUC (Section 25)
# ======================================================================
roc_auc = roc_auc_score(y_test, y_proba)
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

print(f"\nROC-AUC: {roc_auc:.4f}")

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color="#2980b9", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color="gray", lw=1, linestyle="--", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Chart 9: ROC Curve - Logistic Regression", fontsize=13, fontweight="bold")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart9_roc_curve.png")
plt.close()



In [ ]:
# ======================================================================
# 18. FEATURE INTERPRETATION (Section 26)
# ======================================================================
ohe = final_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_feature_names = ohe.get_feature_names_out(categorical_features)
all_feature_names = numeric_features + list(cat_feature_names)
coefs = final_pipeline.named_steps["classifier"].coef_[0]
coef_df = pd.DataFrame({"Feature": all_feature_names, "Coefficient": coefs})
coef_df["AbsCoefficient"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("AbsCoefficient", ascending=False)

print("\n" + "=" * 70)
print("SECTION 26: LOGISTIC REGRESSION COEFFICIENTS (sorted by magnitude)")
print("=" * 70)
print(coef_df[["Feature", "Coefficient"]].to_string(index=False))

plt.figure(figsize=(8, 6))
top_coef = coef_df.head(12).sort_values("Coefficient")
colors = ["#c0392b" if v < 0 else "#27ae60" for v in top_coef["Coefficient"]]
plt.barh(top_coef["Feature"], top_coef["Coefficient"], color=colors)
plt.title("Top Feature Coefficients - Logistic Regression", fontsize=13, fontweight="bold")
plt.xlabel("Coefficient (standardized scale)")
plt.tight_layout()
plt.savefig(f"{CHART_DIR}/chart10_feature_coefficients.png")
plt.close()

# Random Forest feature importance for cross-reference
rf_ohe = rf_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
rf_cat_names = rf_ohe.get_feature_names_out(categorical_features)
rf_all_names = numeric_features + list(rf_cat_names)
rf_importance = rf_pipeline.named_steps["classifier"].feature_importances_
rf_imp_df = pd.DataFrame({"Feature": rf_all_names, "Importance": rf_importance}).sort_values(
    "Importance", ascending=False)
print("\nRandom Forest feature importance (cross-reference):\n", rf_imp_df.to_string(index=False))



In [ ]:
# ======================================================================
# 19. SAMPLE PREDICTIONS (Section 27)
# ======================================================================
sample_passengers = pd.DataFrame([
    {"Pclass": 1, "Sex": "female", "Age": 28, "SibSp": 0, "Parch": 0, "Fare": 80.0,
     "Embarked": "S", "FamilySize": 1, "IsAlone": 1, "Title": "Miss"},
    {"Pclass": 3, "Sex": "male", "Age": 25, "SibSp": 0, "Parch": 0, "Fare": 7.9,
     "Embarked": "S", "FamilySize": 1, "IsAlone": 1, "Title": "Mr"},
    {"Pclass": 2, "Sex": "female", "Age": 4, "SibSp": 1, "Parch": 1, "Fare": 23.0,
     "Embarked": "C", "FamilySize": 3, "IsAlone": 0, "Title": "Miss"},
    {"Pclass": 3, "Sex": "male", "Age": 45, "SibSp": 1, "Parch": 2, "Fare": 15.5,
     "Embarked": "Q", "FamilySize": 4, "IsAlone": 0, "Title": "Mr"},
])
sample_preds = final_pipeline.predict(sample_passengers)
sample_proba = final_pipeline.predict_proba(sample_passengers)[:, 1]
sample_passengers["Predicted_Survived"] = sample_preds
sample_passengers["Survival_Probability"] = sample_proba.round(4)

print("\n" + "=" * 70)
print("SECTION 27: SAMPLE PREDICTIONS")
print("=" * 70)
print(sample_passengers.to_string(index=False))



In [ ]:
# ======================================================================
# 20. EXPORT FILES (Section 35)
# ======================================================================
df.to_csv("titanic_cleaned.csv", index=False)
comparison_df.to_csv("model_comparison.csv", index=False)
coef_df.to_csv("feature_coefficients.csv", index=False)
sample_passengers.to_csv("sample_predictions.csv", index=False)

# Save a results summary for the report-building step
results = {
    "rows": int(df_raw.shape[0]),
    "cols": int(df_raw.shape[1]),
    "missing_age": int(df_raw["Age"].isnull().sum()),
    "missing_cabin": int(df_raw["Cabin"].isnull().sum()),
    "missing_embarked": int(df_raw["Embarked"].isnull().sum()),
    "duplicates": int(n_duplicates),
    "n_survived": n_survived,
    "n_not_survived": n_not_survived,
    "pct_survived": pct_survived,
    "pct_not_survived": pct_not_survived,
    "sex_survival": sex_survival.to_dict(),
    "pclass_survival": pclass_survival.to_dict(),
    "embarked_survival": embarked_survival.to_dict(),
    "family_survival": family_survival.to_dict(),
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "roc_auc": roc_auc,
    "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    "age_median": age_median,
    "embarked_mode": embarked_mode,
}
import json
with open("results_summary.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("\n" + "=" * 70)
print("ALL DONE — files written: titanic_cleaned.csv, model_comparison.csv,")
print("feature_coefficients.csv, sample_predictions.csv, results_summary.json")
print("Charts saved in ./charts/")
print("=" * 70)
